# 14 · Phase 1 analysis and Gate 1  (CPU session)

Tables and figures from the immutable result files, the pre-registered rule checks, and the state metrics (E1.4:
effective rank / Frobenius norm of the GDN state, plus β and Engram-gate α on pair vs filler tokens).
**This notebook does not interpret anything. The Gate 1 decision is yours** (design §7.4).

**Inputs:** every Phase 1 result and final.pt. Attach the outputs of NB10–13 via `PREV_OUTPUTS`, or use `HF_REPO`.

### Logs and downloads (created automatically while this notebook runs)

Everything appears in this notebook's **Output** tab (`/kaggle/working`) and can be downloaded directly:

| File | What it contains |
|---|---|
| `run.log` | **The whole run**: everything this notebook prints, live, with elapsed seconds per line: every command, every training step, every warning and the full traceback of any failed cell |
| `results/_logs/<group>/<run_id>.log` | One log per training run (the full output of that run, both GPUs) |
| `14_phase1_analysis_results.zip` | Rebuilt after every cell: `results/` (result JSONs + all the logs above, no checkpoints). **Download this one** |

To hand the results back: download `14_phase1_analysis_results.zip` and run
`unzip -n 14_phase1_analysis_results.zip -d <your Engram repo>`. If the session dies, `run.log` and the per-run logs
still hold everything up to that moment. The code itself is cloned to `/tmp` and is **not** part of the output.

In [ ]:
# ===================== SETTINGS: edit this cell, then "Run All" (or "Save Version -> Save & Run All") =====================
REPO_URL     = "https://github.com/Shiveshrane/Engram.git"   # None -> copy the code from CODE_DATASET instead
REPO_REF     = "main"           # pin the PREREG commit hash for Phase 1 / 2 runs
CODE_DATASET = "/kaggle/input/engramdelta-src"   # Kaggle Dataset with the repo (used when REPO_URL is None)
HF_REPO      = None             # e.g. "your-hf-user/engramdelta-runs" (private dataset repo; needs Kaggle Secret HF_TOKEN)
GDN_BACKEND  = "torch"          # "torch" (pure PyTorch, always works) or "fla" (only if NB00 says PASS)
FLA_SPEC     = "fla-core==0.5.2"                 # FLA's kernel package (fla.ops); installed with --no-deps
PREV_OUTPUTS = []               # outputs of earlier sessions attached as inputs, e.g. ["/kaggle/input/engramdelta-11-phase1-h1-grid"]
NCCL_ENV     = {}               # set to {"NCCL_P2P_DISABLE": "1", "NCCL_IB_DISABLE": "1"} if NB00's first NCCL test hung

In [ ]:
# ===================== Session setup (no edits needed) =====================
import os, sys, subprocess, shutil, pathlib, json, glob, time, datetime, traceback, zipfile
NEEDS_GPU, INSTALL_FLA = False, False
NB_NAME = "14_phase1_analysis"
ON_KAGGLE = os.path.isdir("/kaggle/working")
WORK = pathlib.Path("/kaggle/working") if ON_KAGGLE else pathlib.Path.cwd()
# On Kaggle the code is cloned to /tmp, which is NOT saved, so the notebook output holds only what this session
# creates: results/, figures/, logs/, ckpt/ (unfinished runs only) and the results zip. results/ and figures/ are
# symlinked from the repo into /kaggle/working. Locally (notebook opened from the repo's notebooks/ folder) the
# repo is the parent directory.
REPO = pathlib.Path("/tmp/engramdelta") if ON_KAGGLE else (WORK.parent if WORK.name == "notebooks" else WORK)

# 0. run.log: everything this notebook prints (every command, every training step, every error traceback) is also
#    written, live, to run.log at the top of the notebook output, with elapsed seconds per line. Each training run
#    also gets its own log, results/_logs/<group>/<run_id>.log. After every cell, results/ (result JSONs + all logs,
#    no checkpoints) is zipped into <notebook>_results.zip. All three are in the Output tab, ready to download.
class _Tee:
    _engram_tee = True
    def __init__(self, stream, f, t0):
        self._s, self._f, self._t0, self._bol = stream, f, t0, True
    def write(self, s):
        out = []
        for part in s.splitlines(keepends=True):
            if self._bol:
                out.append(f"[{time.time() - self._t0:9.1f}s] ")
            out.append(part)
            self._bol = part.endswith("\n")
        self._f.write("".join(out)); self._f.flush()
        return self._s.write(s)
    def flush(self):
        self._s.flush(); self._f.flush()
    def __getattr__(self, k):
        return getattr(self._s, k)

if not getattr(sys.stdout, "_engram_tee", False):      # setup cell re-run: keep the existing log
    _STAMP = datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%dT%H%M%SZ')
    LOG_PATH = WORK / "run.log"
    _LOG = open(LOG_PATH, "a", buffering=1)
    _LOG.write(f"===== {NB_NAME} | started {_STAMP} (UTC) =====\n")
    _T0 = time.time()
    sys.stdout, sys.stderr = _Tee(sys.stdout, _LOG, _T0), _Tee(sys.stderr, _LOG, _T0)
ZIP_PATH = WORK / f"{NB_NAME}_results.zip"

def bundle():
    """Copy the session log into results/_logs/notebooks/ and zip results/ into ZIP_PATH (no checkpoints)."""
    try:
        res = REPO / "results"
        if not res.is_dir():
            return
        dst = res / "_logs" / "notebooks"; dst.mkdir(parents=True, exist_ok=True)
        _LOG.flush(); shutil.copy2(LOG_PATH, dst / f"{NB_NAME}__{_STAMP}.log")
        tmp = ZIP_PATH.with_name(ZIP_PATH.name + ".tmp")
        with zipfile.ZipFile(tmp, "w", zipfile.ZIP_DEFLATED) as z:
            for root, _, files in os.walk(res):
                for fn in files:
                    fp = os.path.join(root, fn); z.write(fp, os.path.relpath(fp, REPO))
        os.replace(tmp, ZIP_PATH)
    except Exception as e:      # never let logging break the experiment
        sys.__stdout__.write(f"!! results zip failed: {e}\n")

_ip = get_ipython()
for _ev in ("pre_run_cell", "post_run_cell"):
    for _cb in list(_ip.events.callbacks[_ev]):
        if getattr(_cb, "__name__", "").startswith("_engram_"):
            _ip.events.unregister(_ev, _cb)
def _engram_pre(info=None):
    src = (getattr(info, "raw_cell", "") or "").strip().splitlines()
    _LOG.write(f"\n===== cell start {time.strftime('%H:%M:%S')} | {src[0][:100] if src else ''}\n")
def _engram_post(result=None):
    err = getattr(result, "error_in_exec", None) or getattr(result, "error_before_exec", None)
    if err is not None:
        _LOG.write("!! CELL ERROR\n" + "".join(traceback.format_exception(type(err), err, err.__traceback__)))
    bundle()
_ip.events.register("pre_run_cell", _engram_pre)
_ip.events.register("post_run_cell", _engram_post)
print("=" * 100)
print(f"LOGS  run.log (this whole notebook, live)  -> {LOG_PATH}")
print(f"      one log per training run             -> {REPO / 'results' / '_logs'}/<group>/<run_id>.log")
print(f"      everything zipped after every cell   -> {ZIP_PATH}")
print("      Download from the notebook's Output tab (run.log and the zip are at the top level).")
print("=" * 100)

def run(cmd, check=True, env=None, timeout=None, show=None):
    """Run a shell command in the repo and stream its output into this notebook."""
    print("$", show or cmd, flush=True)
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
                         env={**os.environ, **(env or {})}, cwd=str(REPO) if REPO.exists() else str(WORK))
    t0 = time.time()
    try:
        for line in p.stdout:
            print(line, end="", flush=True)
            if timeout and time.time() - t0 > timeout:
                raise subprocess.TimeoutExpired(cmd, timeout)
        rc = p.wait(timeout=timeout)
    except subprocess.TimeoutExpired:
        p.kill(); print(f"!! timed out after {timeout}s"); rc = -9
    if check and rc != 0:
        raise RuntimeError(f"exit code {rc}: {show or cmd}")
    return rc

def secret(name):
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret(name)
    except Exception:
        return os.environ.get(name)

# 1. code
if not REPO.exists():
    if REPO_URL:
        url, tok = REPO_URL, secret("GITHUB_TOKEN")
        if tok and url.startswith("https://"):
            url = url.replace("https://", f"https://{tok}@")
        run(f"git clone -q {url} {REPO}", show=f"git clone {REPO_URL}")
        run(f"git -C {REPO} checkout -q {REPO_REF}")
    else:
        shutil.copytree(CODE_DATASET, REPO)
if ON_KAGGLE:
    for d in ("results", "figures"):
        out, link = WORK / d, REPO / d
        out.mkdir(parents=True, exist_ok=True)
        if not link.is_symlink():
            if link.exists():
                shutil.copytree(link, out, dirs_exist_ok=True); shutil.rmtree(link)
            link.symlink_to(out, target_is_directory=True)
os.chdir(REPO); sys.path.insert(0, str(REPO))
try:
    commit = subprocess.run(["git", "rev-parse", "HEAD"], capture_output=True, text=True, check=True).stdout.strip()
except Exception:
    commit = open("GIT_COMMIT").read().strip() if os.path.exists("GIT_COMMIT") else "unknown"
os.environ["GIT_COMMIT"] = commit
print("code commit:", commit)

# 2. packages (never let pip replace Kaggle's torch)
run("pip install -q --no-deps einops sympy")
if INSTALL_FLA or GDN_BACKEND == "fla":
    run(f"pip install -q --no-deps {FLA_SPEC}", check=False)

# 3. environment for every launched run
os.environ["ENGRAMDELTA_GDN_BACKEND"] = GDN_BACKEND
os.environ["ENGRAMDELTA_CKPT_DIR"] = str(WORK / "ckpt")
os.environ.update(NCCL_ENV)
if HF_REPO and secret("HF_TOKEN"):
    os.environ["HF_REPO"], os.environ["HF_TOKEN"] = HF_REPO, secret("HF_TOKEN")
    print("HF Hub sync: ON ->", HF_REPO)
else:
    print("HF Hub sync: OFF (results persist via this notebook's output only)")

# 4. restore results / checkpoints from earlier sessions (never overwrites existing files)
def restore(src, dst):
    if not os.path.isdir(src):
        return 0
    n = 0
    for root, _, files in os.walk(src):
        for f in files:
            s = os.path.join(root, f); d = os.path.join(dst, os.path.relpath(s, src))
            if not os.path.exists(d):
                os.makedirs(os.path.dirname(d), exist_ok=True); shutil.copy2(s, d); n += 1
    return n
for prev in PREV_OUTPUTS:
    n1 = restore(os.path.join(prev, "results"), str(REPO / "results"))
    n1 += restore(os.path.join(prev, "engramdelta", "results"), str(REPO / "results"))   # outputs made before this layout
    n2 = restore(os.path.join(prev, "ckpt"), str(WORK / "ckpt"))
    print(f"restored from {prev}: {n1} result files, {n2} checkpoint files")
if os.environ.get("HF_REPO"):
    run("python -m engramdelta.scripts.sync pull results", check=False)

# 5. hardware
if NEEDS_GPU:
    run("nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv", check=False)
import torch
print("torch", torch.__version__, "| CUDA", torch.version.cuda, "| GPUs:", torch.cuda.device_count())
NPROC = max(1, torch.cuda.device_count())
if NEEDS_GPU and NPROC < 2:
    print("!! WARNING: fewer than 2 GPUs. Set Accelerator = 'GPU T4 x2' in the notebook settings.")

In [ ]:
if os.environ.get("HF_REPO"):
    run("python -m engramdelta.scripts.sync pull ckpt", check=False)
import yaml, pandas as pd
from engramdelta import analysis
pd.set_option("display.width", 200)
pr = yaml.safe_load(open("configs/prereg.yaml"))
df = analysis.load("results")
p1 = df[df["phase"] == "phase1"]
print(f"{len(p1)} finished Phase 1 runs; failures logged:", os.listdir("results/_failures") if os.path.isdir("results/_failures") else "none")
print("fp16 health: runs with non-finite losses:", int((p1["nonfinite_losses"] > 0).sum()),
      "| max GradScaler skipped steps:", p1["skipped_steps"].max())
print("code commits used:", p1["git_commit"].value_counts().to_dict())
print("non-embedding params per arm:", p1.groupby("arm")["non_embedding"].first().to_dict())

## LR pilot (E1.0a) and the λ check (E1.0c vs λ = 1)

In [ ]:
print(analysis.lr_pilot_table(df).to_string(index=False))
print(analysis.lambda_check(df, pr).to_string(index=False))

## H1 grid (A1)

In [ ]:
print(analysis.h1_grid(df).to_string(index=False))

## All arms per cell (λ = 1)

In [ ]:
print(analysis.arms_table(df).to_string(index=False))

## Gate 1 table: pre-registered primary-cell comparisons (rule: |diff| > 2 × pooled std, n ≥ 3)

In [ ]:
print(analysis.gate1_table(df, pr).to_string(index=False))

## H4: 2×2 conv ablation

In [ ]:
print(analysis.h4_table(df, pr).to_string(index=False))

## Leak checks (A6 must be < 1%)

In [ ]:
print(analysis.leak_table(df).to_string(index=False))

## E1.4 state metrics (exploratory)

In [ ]:
run("python -m engramdelta.scripts.run_state_metrics --groups E1.1 E1.2 --arms A1 A2 A3 A4", check=False)
from engramdelta.utils.run_log import read_results
rows = []
for r in read_results("results/phase1/E1.4"):
    d = r["data"]
    for layer, segs in r["state_occupancy"].items():
        for seg, m in segs.items():
            rows.append({"arm": r["arm"], "N": d["N"], "F": d["F"], "P": d["P"], "placement": d["placement"],
                         "layer": layer, "segment": seg, "eff_rank": m["effective_rank"], "fro": m["fro"]})
if rows:
    st = pd.DataFrame(rows)
    print(st.groupby(["N", "F", "placement", "layer", "segment", "arm"])[["eff_rank", "fro"]].agg(["mean", "std"]).to_string())
    kinds = []
    for r in read_results("results/phase1/E1.4"):
        for name, cats in r["token_kind_stats"].items():
            kinds.append({"arm": r["arm"], "F": r["data"]["F"], "placement": r["data"]["placement"], "stat": name, **cats})
    print(pd.DataFrame(kinds).groupby(["stat", "arm", "F", "placement"]).mean(numeric_only=True).to_string())

## Figures

In [ ]:
run("python -m engramdelta.scripts.make_plots --results-dir results --out figures", check=False)
from IPython.display import Image, display
for f in sorted(glob.glob("figures/*.png")):
    display(Image(f))

## Gate 1 (your decision, design §7.4)
Continue to Phase 2 only if H1 holds AND A2 > A1 at the primary cell AND the A3/A4 controls do not explain the gain.
If H1 fails, stop and write up the negative result (or reframe). If A3/A4 explain the gain, reframe as "gated side-pathway".

In [ ]:
# ===================== End of session: persist =====================
if os.environ.get("HF_REPO"):
    run("python -m engramdelta.scripts.sync push results", check=False)
n = sum(len(f) for _, _, f in os.walk(REPO / "results"))
print(f"{n} files under results/. They are part of this notebook's output: Save Version keeps them.")
print("Next session: attach this notebook's output as an input and add its path to PREV_OUTPUTS (or use HF_REPO).")
bundle()
_mb = ZIP_PATH.stat().st_size / 1e6 if ZIP_PATH.exists() else 0.0
print(f"\nTo send the results back: Output tab -> download {ZIP_PATH.name} ({_mb:.1f} MB) "
      "and unzip it into the repo folder:")
print(f"    unzip -n {ZIP_PATH.name} -d <your Engram repo>     # -n: never overwrite existing files")
print("It holds results/ (result JSONs), results/_logs/<group>/<run_id>.log (each run's training log)")
print(f"and results/_logs/notebooks/{NB_NAME}__{_STAMP}.log (a copy of run.log).")
print(f"run.log alone (this notebook's full printed output) is also in the Output tab: {LOG_PATH}")